<a href="https://colab.research.google.com/github/yongik-jang/ml-for-very-newbies/blob/main/notebooks/01_examples_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML for Very Newbies — Worked Examples

This notebook runs **in parallel with the slides**. Each section reproduces, in code,
something the slides only assert. The point is not the code style — it is that you can
check every claim yourself instead of believing it.

| notebook | slides |
|---|---|
| 1. Building Blocks | Perceptron, XOR, MLP, nonlinearity, universal approximation |
| 2. Backpropagation | Forward/backward, the computational graph |
| 3. Optimization | SGD, Momentum, AdaGrad, RMSProp, Adam |
| 4. In Practice | Overfitting, early stopping |

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

print("numpy    ", np.__version__)
print("torch    ", torch.__version__)
print("device   ", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# for reproducibility
SEED = 99

def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed()

In [ ]:
def report(name, a, b):
    '''Print the largest absolute disagreement between two arrays.'''
    a = np.asarray(a, dtype=float).ravel()
    b = np.asarray(b, dtype=float).ravel()
    err = np.max(np.abs(a - b))
    print(f"{name:<44s} max |diff| = {err:.3e}")

---
# 1. Building Blocks

## 1.1 A perceptron, by hand

One unit. A weighted sum, a bias, a step:

$$\hat y = H(\vec w\cdot\vec x + b),\qquad H(z)=\begin{cases}1 & z>0\\ 0 & z\le 0\end{cases}$$

and the perceptron learning rule, which is not gradient descent — the step function has
no useful derivative. It is simply *"if you got it wrong, move the weights toward the
right answer"*:

$$\vec w \leftarrow \vec w + \eta\,(y-\hat y)\,\vec x,\qquad b \leftarrow b + \eta\,(y-\hat y)$$

In [ ]:
def heaviside(z):
    # TODO: 1 if z > 0, else 0 (as float)
    return ...

def perceptron_fit(X, y, epochs=20, lr=0.1, seed=0):
    '''
    Rosenblatt's rule. Returns (w, b, mistakes_per_epoch).
    '''
    rng = np.random.default_rng(seed)
    w = np.zeros(X.shape[1])
    b = 0.
    history = []
    for _ in range(epochs):
        mistakes = 0
        # Shuffle the data indices for each epoch
        for i in rng.permutation(len(X)):
            # TODO: forward pass of one unit
            y_hat = ...
            err = y[i] - y_hat
            # Update only when the prediction is wrong
            if err != 0.:
                # TODO: perceptron learning rule (see the formula above)
                w += ...
                b += ...
                mistakes += 1
        history.append(mistakes)
    return w, b, np.array(history)

In [ ]:
# the four points of the boolean plane
X4 = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y_AND = np.array([0., 0., 0., 1.])
y_OR  = np.array([0., 1., 1., 1.])
y_XOR = np.array([...])   # TODO: XOR of (x1, x2) for the four rows of X4

In [ ]:
w, b, hist = perceptron_fit(X4, y_AND, epochs=20, lr=0.1)
print(f"AND:  w = {w},  b = {b:.2f}")
print(f"      mistakes per epoch: {hist}")
print()
for x, t in zip(X4, y_AND):
    print(f"      x = {x}  ->  y_hat = {heaviside(x @ w + b):.0f}   (target {t:.0f})")

In [ ]:
_, _, hist_and = perceptron_fit(X4, y_AND, epochs=30, lr=0.1)
_, _, hist_or  = perceptron_fit(X4, y_OR,  epochs=30, lr=0.1)
_, _, hist_xor = perceptron_fit(X4, y_XOR, epochs=30, lr=0.1)

plt.plot(hist_and, lw=3, label="AND")
plt.plot(hist_or, lw=3, ls='--', label="OR")
plt.plot(hist_xor, lw=3, label="XOR")

plt.xlabel("Epoch", fontsize=15) 
plt.ylabel("Mistakes", fontsize=15)
plt.yticks([0, 1, 2, 3, 4])

plt.grid(ls='--')
plt.legend(fontsize=15)

plt.tight_layout()
plt.show()

print("mistakes in the last 10 epochs")
print(f"  AND {hist_and[-10:]}")
print(f"  OR  {hist_or[-10:]}")
print(f"  XOR {hist_xor[-10:]}   <- it keeps oscillating forever")

This is not a failure of the learning rule. Rosenblatt's convergence theorem guarantees
that it finds a separating line **if one exists**. For XOR none exists, so there is
nothing to find. Let us check that claim by brute force rather than taking it on faith.

In [ ]:
# Scan a fine grid of lines w1*x1 + w2*x2 + b = 0 and ask whether ANY of them
# classifies all four XOR points correctly.
grid = np.linspace(-3, 3, 61)
best = 0
for w1 in grid:
    for w2 in grid:
        for bb in grid:
            pred = heaviside(X4 @ np.array([w1, w2]) + bb)
            best = max(best, int((pred == y_XOR).sum()))

print(f"exhaustive scan over {len(grid)**3:,} lines")
print(f"best score on XOR: {best} of 4 points")
print("A single linear boundary tops out at 3 of 4. No line separates XOR.")

## 1.2 Hidden space

The trick is not a better line — it is better coordinates. Take the two hidden units from
the slide,

$$h_1 = H(x_1+x_2-1.5)\qquad h_2 = H(x_1+x_2-0.5)$$

and look at where the four input points land.

In [ ]:
def hidden_map(X):
    # TODO: the two hidden units from the slide
    h1 = ...
    h2 = ...
    return np.column_stack([h1, h2])

Hpts = hidden_map(X4)

print("   x            ->   (h1, h2)     label")
for x, h, t in zip(X4, Hpts, y_XOR):
    print(f"  ({x[0]:.0f}, {x[1]:.0f})        ->   ({h[0]:.0f}, {h[1]:.0f})          {t:.0f}")
print()
print("(0,1) and (1,0) -- the two points with label 1 -- land on the SAME point.")
print("Four points became three, and three points are always separable.")

In [ ]:
# the same perceptron, now trained in hidden coordinates
w_h, b_h, hist_h = perceptron_fit(Hpts, y_XOR, epochs=20, lr=0.1)
print(f"w = {w_h}, b = {b_h:.2f}")
print(f"mistakes per epoch: {hist_h}")

## 1.3 An MLP finds its own hidden space

We handed the network those two hidden units. A real network is not told what they
should be — it discovers them. Here is the first appearance of PyTorch, and of
`autograd`: we write only the **forward** pass and the loss.

In [ ]:
set_seed(10)

Xt = torch.tensor(X4, dtype=torch.float32)
yt = torch.tensor(y_XOR, dtype=torch.float32).unsqueeze(1)

# TODO: 2 inputs -> 2 hidden units (tanh) -> 1 output (a logit, not a probability)
mlp = nn.Sequential(
    ...,
    ...,
    ...,
)

opt = torch.optim.Adam(mlp.parameters(), lr=0.08)
loss_fn = nn.BCEWithLogitsLoss()

losses = []
for step in range(3000):
    # TODO: the five lines of every training loop
    ...        # 1. clear old gradients
    logit = ...    # 2. forward
    loss = ...     # 3. loss
    ...        # 4. backward
    ...        # 5. update
    losses.append(loss.item())

print(f"final loss: {losses[-1]:.2e}\n")
with torch.no_grad():
    prob = torch.sigmoid(mlp(Xt)).squeeze(1)
    hid  = torch.tanh(mlp[0](Xt))
print("   x         y     y_hat      learned (h1, h2)")
for x, t, p, h in zip(X4, y_XOR, prob, hid):
    print(f"  ({x[0]:.0f}, {x[1]:.0f})     {t:.0f}    {p:.4f}    ({h[0]:+.2f}, {h[1]:+.2f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].semilogy(losses, color='red', lw=3)
axes[0].set_xlabel("Epoch", fontsize=15)
axes[0].set_ylabel("BCE loss", fontsize=15)
axes[0].set_title("Training loss", fontsize=15)
axes[0].grid(ls='--')

gx, gy = np.meshgrid(np.linspace(-0.4, 1.4, 250), np.linspace(-0.4, 1.4, 250))
with torch.no_grad():
    G = torch.tensor(np.column_stack([gx.ravel(), gy.ravel()]), dtype=torch.float32)
    P = torch.sigmoid(mlp(G)).numpy().reshape(gx.shape)
im = axes[1].contourf(gx, gy, P, levels=np.linspace(0, 1, 21), cmap="RdYlBu_r", alpha=0.85)
axes[1].contour(gx, gy, P, levels=[0.5], colors="black", linewidths=2)
for x, t in zip(X4, y_XOR):
    axes[1].scatter(*x, s=200, color='blue' if x[0] == x[1] else 'red',
                    edgecolor="white", linewidth=1.6, zorder=3)
axes[1].set_xlabel(r"$x_1$", fontsize=15)
axes[1].set_ylabel(r"$x_2$", fontsize=15)
axes[1].set_title("Learned decision surface", fontsize=15)
cbar = fig.colorbar(im, ax=axes[1])
cbar.set_label(label=r"$\hat y$", size=15)
plt.tight_layout(); plt.show()

The boundary is curved, and the two hidden units did **not** converge to our hand-picked
$h_1,h_2$. They found some other pair that works just as well. There is no unique
solution — a point worth making out loud when someone asks what a hidden unit "means".

## 1.4 Universal approximation

The theorem says a single hidden layer, wide enough, approximates any continuous function
on a compact set to any accuracy. Let us watch the width do the work.

In [ ]:
def target_fn(x):
    return np.sin(3 * x) * np.exp(-x**2 / 8) + 0.15 * x

def fit_width(X , Y, n_hidden, steps=1000, lr=0.02, seed=0):
    set_seed(seed)
    # TODO: one hidden layer of width n_hidden, tanh activation, scalar output
    net = nn.Sequential(
        ...,
        ...,
        ...
        )
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    loss_fn = ...   # TODO: regression loss
    for _ in range(steps):
        opt.zero_grad()
        y_hat = net(X)
        loss = loss_fn(y_hat, Y)
        loss.backward()
        opt.step()
    return net, loss.item()

In [ ]:
x_train = np.linspace(-3, 3, 200)
y_train = target_fn(x_train)

Xtr = torch.tensor(x_train, dtype=torch.float32).unsqueeze(1)
Ytr = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

widths = [1, 3, 10, 64]
fits = {}
for n in widths:
    net, final = fit_width(Xtr, Ytr, n)
    fits[n] = net
    print(f"hidden units = {n:>3d}   final MSE = {final:.2e}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True, sharey=True)
for ax, n in zip(axes.ravel(), widths):
    with torch.no_grad():
        pred = fits[n](Xtr).squeeze(1).numpy()
    ax.plot(x_train, y_train, color='blue', lw=5, alpha=0.3, label="target")
    ax.plot(x_train, pred, color='red', lw=2, ls="--", label="network")
    ax.set_title(f"{n} hidden unit" + ("s" if n > 1 else ""), fontsize=15)
    if n == widths[-1]:
        ax.legend(loc="lower right", fontsize=14, frameon=False)
for ax in axes[1]:
    ax.set_xlabel("x", fontsize=15)
plt.tight_layout()
plt.show()

### What the theorem does not say

It is a statement about approximating a function **on the interval used for training**. 
Step outside that interval, and the guarantee is simply absent. 
This is the same idea as overfitting.

In [ ]:
x_wide = np.linspace(-7, 7, 400)
with torch.no_grad():
    pred_wide = fits[64](torch.tensor(x_wide, dtype=torch.float32).unsqueeze(1)).squeeze(1).numpy()

fig, ax = plt.subplots(figsize=(8, 5))
ax.axvspan(-3, 3, color='gray', alpha=0.3, lw=0)
ax.plot(x_wide, target_fn(x_wide), color='blue', lw=3, label="target")
ax.plot(x_wide, pred_wide, color='red', lw=1.8, ls="--", label="network (64 units)")
ax.text(0, 1.45, "trained here", ha="center", color='gray', fontsize=15)
ax.set_xlabel("x", fontsize=15)
ax.set_ylim(-2.0, 1.7)
ax.legend(loc="lower left", frameon=False, fontsize=15)
plt.show()

inside  = np.abs(pred_wide - target_fn(x_wide))[(x_wide > -3) & (x_wide < 3)].mean()
outside = np.abs(pred_wide - target_fn(x_wide))[(x_wide < -3) | (x_wide > 3)].mean()
print(f"mean |error| inside  [-3, 3] : {inside:.4f}")
print(f"mean |error| outside [-3, 3] : {outside:.4f}")

---
# 2. Backpropagation

Backpropagation is the chain rule plus one accounting trick: compute each shared
sub-expression once and reuse it.

## 2.1 autograd
### PyTorch `autograd`

PyTorch `autograd` automatically computes gradients by tracking the **computational graph**.

```python
z = torch.tensor(2.0, requires_grad=True)
a = z ** 2

g, = torch.autograd.grad(a, z)
```

`requires_grad=True` tells PyTorch to track operations involving `z`.

Here,

$$
a=z^2
$$

so

$$
g=\frac{da}{dz}=2z=4
$$

---

If `a` is a vector, we can use:

```python
g, = torch.autograd.grad(a.sum(), z)
```

`a.sum()` turns the vector output into a scalar. If \(a_i=f(z_i)\),

$$
\frac{\partial}{\partial z_i}\sum_j a_j
=
f'(z_i)
$$

so `g` contains the derivative \(f'(z_i)\) at each input point.

`torch.autograd.grad()` returns a tuple:

```python
(tensor([...]),)
```

so

```python
g, = ...
```

is tuple unpacking and is equivalent to:

```python
g = result[0]
```

---

`retain_graph` controls whether the computational graph is kept after computing the gradient.

```python
retain_graph=False  # default
```

The graph is freed after the gradient is computed.

```python
g1, = torch.autograd.grad(a, z, retain_graph=True)
g2, = torch.autograd.grad(a, z)
```

With `True`, the **same graph can be reused**. It does not change the gradient value.

In normal training, `retain_graph=True` is usually unnecessary.

Finally,

```python
z.detach()
```

returns a tensor detached from the computational graph, which is useful when we only need the values, such as for plotting.

$$
\boxed{
\texttt{requires\_grad}
\rightarrow
\text{computational graph}
\rightarrow
\texttt{autograd.grad()}
\rightarrow
\text{gradient}
}
$$


In [ ]:
x = torch.tensor(3.0, dtype=torch.float32, requires_grad=True)
y = torch.tensor(4.0, dtype=torch.float32, requires_grad=True)

f = x**2 * y + torch.sin(x)
f_dx, f_dy = torch.autograd.grad(f, (x, y), create_graph=True)
f_dxdy, = torch.autograd.grad(f_dx, y)

print(f"f = x^2 y + sin(x)  at (3, 4) = {f.item():.6f}")
# TODO: fill in the analytic values at (x, y) = (3, 4) and compare with autograd
dfdx_hand   = ...
dfdy_hand   = ...
dfdxdy_hand = ...
print(f"df/dx     by hand = {dfdx_hand:.6f}   autograd: {f_dx.item():.6f}")
print(f"df/dy     by hand = {dfdy_hand:.6f}   autograd: {f_dy.item():.6f}")
print(f"d^2f/dxdy by hand = {dfdxdy_hand:.6f}   autograd: {f_dxdy.item():.6f}")

In [ ]:
# Three things to be cautious about

# (1) .grad ACCUMULATES. backward() adds to it; it does not overwrite.
# autograd.grad() does not accumulate.

a = torch.tensor(2.0, requires_grad=True)

(a**2).backward()
print(f"after one backward : a.grad = {a.grad.item()}")

(a**2).backward()
print(f"after two backwards: a.grad = {a.grad.item()}  <- 4 + 4, not 4")

a.grad = None
print("this is why every training loop starts with opt.zero_grad()\n")

g, = torch.autograd.grad(a**2, a)
print(f"Autograd.grad(): \n{g.item()}")

g, = torch.autograd.grad(a**2, a)
print(f"{g.item()}\n")


# (2) no_grad() switches the graph off -- use it for evaluation.
w = torch.tensor([1.0, 2.0], requires_grad=True)
with torch.no_grad():
    out = (w * 3).sum()
print(f"inside no_grad, out.requires_grad = {out.requires_grad}\n")


# (3) detach() cuts the graph at one point without switching anything off.
u = torch.tensor(5.0, requires_grad=True)
v = (u * 2).detach() * u          # gradient flows only through the second factor
v.backward()
print(f"v = detach(2u) * u,  dv/du = {u.grad.item()}   (10, not 20)")

## 2.2 One chain, node by node

This diagram represents the sigmoid and the binary cross-entropy written out
as elementary operations, for the case $y=1$.

$$z \xrightarrow{\ \times(-1)\ } u_1 \xrightarrow{\ \exp\ } u_2 \xrightarrow{\ +1\ } u_3
\xrightarrow{\ 1/\cdot\ } \hat y \xrightarrow{\ \log\ } u_5 \xrightarrow{\ \times(-1)\ } \mathcal{L}$$

We walk it backwards by hand, multiplying one local derivative at a time, and then ask
`autograd` whether we got it right.

In [ ]:
z_val = -1.3          # any logit you like
y_val = 1.0

# ---- forward, node by node --------------------------------------------------
u1 = -z_val
u2 = np.exp(u1)
u3 = 1.0 + u2
yh = ...                    # TODO: this is sigmoid(z)
u5 = ...
L  = ...                         # TODO: this is BCE for y = 1

print(f"forward:\n  z  = {z_val}")
print(f"  u1 = -z        = {u1:+.6f}")
print(f"  u2 = exp(u1)   = {u2:+.6f}")
print(f"  u3 = 1 + u2    = {u3:+.6f}")
print(f"  y_hat = 1/u3   = {yh:+.6f}    (= sigmoid(z))")
print(f"  u5 = log(y_hat)= {u5:+.6f}")
print(f"  L  = -u5       = {L:+.6f}")

In [ ]:
# ---- backward, node by node -------------------------------------------------
# Each row: the local derivative of this node's output w.r.t. its input,
# and the running product accumulated so far.
rows = []

g = 1.0                                   ; rows.append(("dL/dL",      "1",            g))
# TODO: multiply by the local derivative of each node, from the loss back to z
g = g * (...)   ; rows.append(("dL/du5",     "?",            g))   # L  = -u5
g = g * (...)   ; rows.append(("dL/dy_hat",  "?",            g))   # u5 = log(y_hat)
g = g * (...)   ; rows.append(("dL/du3",     "?",            g))   # y_hat = 1/u3
g = g * (...)   ; rows.append(("dL/du2",     "?",            g))   # u3 = 1 + u2
g = g * (...)   ; rows.append(("dL/du1",     "?",            g))   # u2 = exp(u1)
g = g * (...)   ; rows.append(("dL/dz",      "?",            g))   # u1 = -z

print(f"{'accumulated':<12s} {'local factor':<12s} {'value':>12s}")
print("-" * 38)
for name, local, val in rows:
    print(f"{name:<12s} {local:<12s} {val:>12.8f}")

print()
print(f"hand-derived  dL/dz = {g:.10f}")
print(f"y_hat - y           = {yh - y_val:.10f}")

z_t = torch.tensor(z_val, requires_grad=True)
F.binary_cross_entropy_with_logits(z_t, torch.tensor(y_val)).backward()
print(f"autograd      dL/dz = {z_t.grad.item():.10f}")
report("binary chain by hand vs autograd", g, z_t.grad.item())

---
# 3. Optimization

Backpropagation tells us $\nabla_\theta\mathcal{L}$. What we do with it is a separate
question, and it is the one that decides whether training takes an hour or a week.

Every optimizer below is written as a plain function of `(params, grads, state)` — no
PyTorch — so that the update rule is visible. At the end we check the hand-written Adam
against `torch.optim.Adam`.

## 3.1 Gradient descent on a bowl

$$f(x,y)=\frac{x^2}{20}+y^2,\qquad \nabla f=\left(\frac{x}{10},\,2y\right)$$

The 20 makes it anisotropic: very gentle along $x$, steep along $y$. That is a cartoon of
the real problem — the curvature differs by orders of magnitude between directions.

In [ ]:
f_bowl    = lambda x, y: x**2 / 20 + y**2
grad_bowl = lambda p: np.array([..., ...])   # TODO: gradient of f_bowl at p = (x, y)

P0 = np.array([-7.0, 2.0])

In [ ]:
def gd(lr, n_steps=30, p0=P0):
    p = p0
    path = [p]
    for _ in range(n_steps):
        p = ...   # TODO: one gradient-descent step
        path.append(p)
    return np.array(path)

In [ ]:
def plot_paths(paths_by_name, l=2):
    from matplotlib.colors import LinearSegmentedColormap
    cmap = LinearSegmentedColormap.from_list(
        "bowl", ["#FFFFFF", "#EDF3F1", "#D5E4E0", "#B9D0CB"])
    X, Y = np.meshgrid(np.linspace(-10, 10, 300), np.linspace(-5, 5, 300))
    Z = np.minimum(f_bowl(X, Y), 7)
    lv = np.linspace(0, 7, 14)

    n = len(paths_by_name)
    fig, axes = plt.subplots(l, 2, figsize=(10, 7) if l > 1 else (14, 5), squeeze=False)
    for ax, (name, path) in zip(axes.ravel(), paths_by_name.items()):
        ax.contourf(X, Y, Z, levels=lv, cmap=cmap, zorder=0)
        ax.contour(X, Y, Z, levels=lv, colors="#9CB7B2", linewidths=0.6, zorder=1)
        ax.plot(path[:, 0], path[:, 1], color='gray', lw=1.6, zorder=3)
        ax.scatter(path[:, 0], path[:, 1], s=14, color='gray', zorder=4, ec="white", linewidths=0.5)
        ax.scatter([0], [0], marker="*", s=200, color='red', zorder=5)
        ax.set_title(name, fontsize=15)
        ax.set_xlim(-10, 10)
        ax.set_ylim(-5, 5)
        ax.set_xticks([])
        ax.set_yticks([])

    plt.tight_layout()
    plt.show()

In [ ]:
plot_paths({fr"$\eta$ = {lr}": gd(lr) for lr in [0.1, 0.5, 0.95, 1.05]})

## 3.2 Five update rules, written out

Using the notation on the slides:

$$
\begin{aligned}
\textbf{SGD}\quad     &\theta \leftarrow \theta - \eta\,g \\[2pt]
\textbf{Momentum}\quad&m \leftarrow \beta_1 m + (1-\beta_1) g, &&\theta \leftarrow \theta - \eta\,m \\[2pt]
\textbf{AdaGrad}\quad &v \leftarrow v + g^2, &&\theta \leftarrow \theta - \eta\,\frac{g}{\sqrt{v}+\varepsilon} \\[2pt]
\textbf{RMSProp}\quad &v \leftarrow \beta_2 v + (1-\beta_2) g^2, &&\theta \leftarrow \theta - \eta\,\frac{g}{\sqrt{v}+\varepsilon} \\[2pt]
\textbf{Adam}\quad    &m \leftarrow \beta_1 m + (1-\beta_1) g,\ \ v \leftarrow \beta_2 v + (1-\beta_2) g^2, &&\theta \leftarrow \theta - \eta\,\frac{\hat m}{\sqrt{\hat v}+\varepsilon}
\end{aligned}
$$

Adam is literally the two lines above it, stacked, plus the bias correction
$\hat m = m/(1-(\beta_1)^t)$, $\hat v = v/(1-(\beta_2)^t)$.
Here, 
$g=\nabla_\theta\mathcal{L}$.

In [ ]:
def make_optimizer(kind, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    '''Returns step(p, g) -> p_new. State is captured in the closure.'''
    state = {"m": None, "v": None, "t": 0}

    def step(p, g):
        if state["m"] is None:
            state["m"] = np.zeros_like(p)
            state["v"] = np.zeros_like(p)
        state["t"] += 1
        t = state["t"]

        if kind == "sgd":
            return p - lr * g

        if kind == "momentum":
            # TODO: running average of gradients, then step along it
            state["m"] = ...
            return ...

        if kind == "adagrad":
            # TODO: accumulate squared gradients, divide the step by sqrt(v) + eps
            state["v"] = ...
            return ...

        if kind == "rmsprop":
            # TODO: same as AdaGrad, but v is a moving average
            state["v"] = ...
            return ...

        if kind == "adam":
            state["m"] = beta1 * state["m"] + (1 - beta1) * g
            state["v"] = beta2 * state["v"] + (1 - beta2) * g * g
            # TODO: bias correction, then the Adam step
            m_hat = ...
            v_hat = ...
            return ...

        raise ValueError(kind)

    return step

def run_optimizer(kind, lr, n_steps=30, p0=P0, **kw):
    step = make_optimizer(kind, lr, **kw)
    p = p0.copy()
    path = [p.copy()]
    for _ in range(n_steps):
        p = step(p, grad_bowl(p))
        path.append(p.copy())
    return np.array(path)

# learning rates chosen per method -- see the comment after the figure
RUNS = [("SGD", "sgd", 0.95), ("Momentum", "momentum", 0.95),
        ("AdaGrad", "adagrad", 1.5), ("Adam", "adam", 0.30)]

plot_paths({name: run_optimizer(kind, lr) for name, kind, lr in RUNS})

for name, kind, lr in RUNS:
    path = run_optimizer(kind, lr)
    print(f"{name:<10s} lr = {lr:<5.2f}  final = ({path[-1][0]:+.4f}, {path[-1][1]:+.4f})")

### 3.2.1 What the bias correction is for

At $t=1$, $m = (1-\beta_1)g = 0.1g$ — the running average starts at zero and is therefore
badly biased toward zero for the first few dozen steps. Dividing by $1-\beta_1^t$ undoes
exactly that. It matters most for $v$, where $\beta_2=0.999$ means the bias persists for
*thousands* of steps.

In [ ]:
beta1, beta2 = 0.9, 0.999
t = np.arange(1, 4001)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(t[:60], 1 - beta1 ** t[:60], color='RED', lw=2.0, label=r"$1-\beta_1^t$")
axes[0].plot(t[:60], 1 - beta2 ** t[:60], color='BLUE', lw=2.0, label=r"$1-\beta_2^t$")
axes[0].axhline(1.0, color='gray', lw=2.0, ls='--')
axes[0].set_xlabel("step t", fontsize=15)
axes[0].set_title("first 60 steps", fontsize=15)
axes[0].set_ylabel("correction denominator", fontsize=15)

axes[1].plot(t, 1 - beta1 ** t, color='RED', lw=2.0, label=r"$1-\beta_1^t$")
axes[1].plot(t, 1 - beta2 ** t, color='BLUE', lw=2.0, label=r"$1-\beta_2^t$")
axes[1].axhline(1.0, color='gray', lw=2.0, ls='--')
axes[1].set_xlabel("step t", fontsize=15)
axes[1].set_title("first 4000 steps", fontsize=15)
axes[1].legend(fontsize=15, frameon=False)
for ax in axes:
    ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
# What happens without it: run Adam with and without bias correction.
def run_adam_nobias(lr, n_steps=30, beta1=0.9, beta2=0.999, eps=1e-8, p0=P0):
    p = p0.copy(); m = np.zeros_like(p); v = np.zeros_like(p)
    path = [p.copy()]
    for _ in range(n_steps):
        g = grad_bowl(p)
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g * g
        p = p - lr * m / (np.sqrt(v) + eps)      # no /(1-beta^t)
        path.append(p.copy())
    return np.array(path)

plot_paths({"Adam (with bias correction)": run_optimizer("adam", 0.30),
            "Adam (without)":              run_adam_nobias(0.30)}, l=1)

---
# 4. In Practice

## 4.1 Overfitting

In [ ]:
set_seed(4)
n_train, n_val = 40, 400
noise = 0.3

x_tr = np.sort(np.random.uniform(-3, 3, n_train))
y_tr = target_fn(x_tr) + noise * np.random.randn(n_train)
x_va = np.linspace(-3, 3, n_val)
y_va = target_fn(x_va) + noise * np.random.randn(n_val)

Xtr_ = torch.tensor(x_tr, dtype=torch.float32).unsqueeze(1)
Ytr_ = torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1)
Xva_ = torch.tensor(x_va, dtype=torch.float32).unsqueeze(1)
Yva_ = torch.tensor(y_va, dtype=torch.float32).unsqueeze(1)

In [ ]:
net = nn.Sequential(
    nn.Linear(1, 256),
    nn.ReLU(),
    nn.Linear(256, 256),
    nn.ReLU(),
    nn.Linear(256, 1)
    )
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

tr_hist, va_hist, snapshots = [], [], {}
EPOCHS = 4000
for ep in range(EPOCHS):
    net.train()
    opt.zero_grad()
    pred = net(Xtr_)
    loss = loss_fn(pred, Ytr_)
    loss.backward()
    opt.step()

    net.eval()
    with torch.no_grad():
        pred = net(Xva_)
        va = loss_fn(pred, Yva_)
    tr_hist.append(loss.item())
    va_hist.append(va.item())
    snapshots[ep] = pred.squeeze(1).numpy().copy()

best_ep = ...   # TODO: early stopping = the epoch with the lowest validation loss
print(f"lowest validation loss {va_hist[best_ep]:.4f} at epoch {best_ep}")
print(f"training loss there    {tr_hist[best_ep]:.4f}")
print(f"at the last epoch:  train {tr_hist[-1]:.5f}   val {va_hist[-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].axvspan(best_ep, EPOCHS, color='gray', alpha=0.3, lw=0)
axes[0].semilogy(tr_hist, color='blue', lw=2, label='training loss')
axes[0].semilogy(va_hist, color='orange', lw=2, label='validation loss')
axes[0].axvline(best_ep, color='gray', ls='--', lw=2)
axes[0].scatter(best_ep, va_hist[best_ep], s=300, marker='*', zorder=5, color='red', label='early stopping')
axes[0].text(best_ep+600, max(va_hist) * 0.9, r"$\longrightarrow$overfitting",
             color='gray', ha="center", fontsize=15)
axes[0].set_xlabel("Epoch", fontsize=15)
axes[0].set_ylabel("MSE", fontsize=15)
axes[0].legend(fontsize=13, frameon=False)

axes[1].plot(x_va, target_fn(x_va), color='blue', lw=5, alpha=0.3, label="truth")
axes[1].scatter(x_tr, y_tr, s=28, color='gray', zorder=3, label="training data (40 points)")
axes[1].plot(x_va, snapshots[best_ep], color='green', lw=3, label=f"epoch {best_ep}")
axes[1].plot(x_va, snapshots[EPOCHS - 1], color='red', lw=3, label=f"epoch {EPOCHS}")
axes[1].set_xlabel("x", fontsize=15)
axes[1].legend(fontsize=13, frameon=False)

plt.tight_layout()
plt.show()